# NB01c — Three-Cuisine Acquisition and Unified LanceDB Rebuild



1. acquire the comparison cuisines from Yelp (`american_new` + `mexican`)
2. stage the American menu-image corpus (Yelp + Roboflow)
3. rebuild LanceDB with a unified `cuisine_type` field across restaurants and reviews
4. create the `american_menus` metadata table


| Output | Path |
|---|---|
| Comparison restaurants parquet | `data/processed/comparison_restaurants.parquet` |
| Comparison reviews parquet | `data/processed/comparison_reviews.parquet` |
| Comparison Yelp menu photos | `data/raw/menu_images_comparison/` |
| Roboflow corpus root | `data/raw/roboflow/ocr_menu/` |
| Unified LanceDB tables | `data/lancedb/` |

Run cells top-to-bottom. Long-running cells are marked clearly. Inspect each gate before moving on.

## 1. Environment setup

Imports, project-root resolution, and preflight checks. This cell confirms the Italian baseline from NB01/NB01b exists before we build the comparison slice on top.

In [ ]:
from pathlib import Path
import json
import tarfile
import zipfile
from collections import defaultdict

import numpy as np
import pandas as pd
import pyarrow as pa
import lancedb
from PIL import Image, UnidentifiedImageError
from tqdm.auto import tqdm
from llama_cpp import Llama

ROOT = Path().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "configs").exists(), f"Cannot find project root from {Path().resolve()}"

RAW       = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
YELP_DIR  = RAW / "yelp"
DB_PATH   = ROOT / "data" / "lancedb"

YELP_BUSINESS = YELP_DIR / "yelp_academic_dataset_business.json"
YELP_REVIEWS  = YELP_DIR / "yelp_academic_dataset_review.json"
ITALIAN_RESTAURANTS = PROCESSED / "restaurants.parquet"
ITALIAN_REVIEWS     = PROCESSED / "reviews.parquet"
PHOTO_TAR           = RAW / "menu_images" / "yelp_photos.tar"
PHOTO_JSON          = RAW / "menu_images" / "photos.json"

for path in [YELP_BUSINESS, YELP_REVIEWS, ITALIAN_RESTAURANTS, ITALIAN_REVIEWS]:
    assert path.exists(), f"Missing prerequisite file: {path}"

print(f"Project root            : {ROOT}")
print(f"Yelp business file      : {YELP_BUSINESS}")
print(f"Yelp review file        : {YELP_REVIEWS}")
print(f"Italian restaurants     : {ITALIAN_RESTAURANTS}")
print(f"Italian reviews         : {ITALIAN_REVIEWS}")
print(f"LanceDB path            : {DB_PATH}")
print(f"Photo tar present       : {PHOTO_TAR.exists()}")
print(f"Photo manifest present  : {PHOTO_JSON.exists()}")

## 2. Configuration

Central place for the comparison-cuisine targets, review cap, model path, and staging directories. Keeping this in one cell prevents both overengineering and scattershot edits later in the notebook.

In [ ]:
METROS = {
    "Philadelphia": "PA",
    "Tampa": "FL",
}

CUISINE_MAP = {
    "american_new": "American (New)",
    "mexican": "Mexican",
}

REVIEW_CAP = 200
REVIEW_BATCH_SIZE = 512

COMPARISON_RESTAURANTS_OUT = PROCESSED / "comparison_restaurants.parquet"
COMPARISON_REVIEWS_OUT     = PROCESSED / "comparison_reviews.parquet"
COMPARISON_PHOTO_ROOT      = RAW / "menu_images_comparison"
ROBOFLOW_ROOT              = RAW / "roboflow" / "ocr_menu"
ROBOFLOW_ZIP               = RAW / "roboflow" / "ocr_menu.zip"

MODEL_PATH   = ROOT / "models" / "embeddinggemma-300M-BF16.gguf"
N_CTX        = 512
N_GPU_LAYERS = -1

print("Metros:")
for city, state in METROS.items():
    print(f"  {city}, {state}")

print("\nCuisine targets:")
for slug, label in CUISINE_MAP.items():
    print(f"  {slug:<13} {label}")

print(f"\nReview cap per comparison restaurant : {REVIEW_CAP}")
print(f"Comparison restaurants out            : {COMPARISON_RESTAURANTS_OUT}")
print(f"Comparison reviews out                : {COMPARISON_REVIEWS_OUT}")
print(f"Comparison photo root                 : {COMPARISON_PHOTO_ROOT}")
print(f"Roboflow root                         : {ROBOFLOW_ROOT}")
print(f"Embedding model                       : {MODEL_PATH.name}")

## 3. Helper functions

Small, single-purpose helpers only. These keep the main cells readable without introducing extra abstraction that the notebook does not need.

In [ ]:
def split_categories(value):
    if not isinstance(value, str) or not value.strip():
        return []
    return [part.strip() for part in value.split(",") if part.strip()]


def has_exact_category(value, target):
    return target in split_categories(value)


def extract_price_range(attributes):
    if not isinstance(attributes, dict):
        return None
    return attributes.get("RestaurantsPriceRange2")


def ensure_photo_manifest():
    if PHOTO_JSON.exists():
        return PHOTO_JSON
    if not PHOTO_TAR.exists():
        raise FileNotFoundError(
            "Neither photos.json nor yelp_photos.tar is present. "
            "Place the Yelp photo archive in data/raw/menu_images/ first."
        )
    print("Extracting photos.json from yelp_photos.tar (one-time)...")
    with tarfile.open(PHOTO_TAR, "r:gz") as tf:
        member = tf.getmember("photos.json")
        data = tf.extractfile(member).read()
    PHOTO_JSON.write_bytes(data)
    return PHOTO_JSON


def collect_image_paths(root):
    exts = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
    return sorted(
        p for p in root.rglob("*")
        if p.is_file() and p.suffix.lower() in exts
    )


def scan_image_metadata(paths, source, cuisine_type, business_id=None):
    rows = []
    unreadable = []
    for path in tqdm(paths, desc=f"Scanning {source}", unit="img"):
        try:
            with Image.open(path) as im:
                width, height = im.size
        except (UnidentifiedImageError, OSError) as e:
            unreadable.append((path, str(e)))
            continue
        rows.append({
            "source": source,
            "cuisine_type": cuisine_type,
            "business_id": business_id,
            "path": str(path),
            "filename": path.name,
            "width": width,
            "height": height,
            "size_kb": round(path.stat().st_size / 1024, 1),
        })
    return rows, unreadable


## 4. Extract comparison restaurants from Yelp business data

This cell streams the Yelp business file once and keeps only restaurants in the two target metros where the category list contains either `American (New)` or `Mexican` as an exact Yelp category token.

In [ ]:
rows = []

with open(YELP_BUSINESS, encoding="utf-8") as f:
    for line in tqdm(f, desc="Filtering businesses", unit=" rows"):
        rec = json.loads(line)

        city  = rec.get("city")
        state = rec.get("state")
        if city not in METROS or METROS[city] != state:
            continue

        categories = rec.get("categories")
        matched = None
        for cuisine_type, yelp_label in CUISINE_MAP.items():
            if has_exact_category(categories, yelp_label):
                matched = cuisine_type
                break

        if matched is None:
            continue

        rows.append({
            "business_id":  rec.get("business_id"),
            "name":         rec.get("name"),
            "address":      rec.get("address"),
            "city":         city,
            "state":        state,
            "postal_code":  rec.get("postal_code"),
            "latitude":     rec.get("latitude"),
            "longitude":    rec.get("longitude"),
            "stars":        rec.get("stars"),
            "review_count": rec.get("review_count"),
            "is_open":      rec.get("is_open"),
            "price_range":  extract_price_range(rec.get("attributes")),
            "categories":   categories,
            "hours":        rec.get("hours"),
            "cuisine_type": matched,
        })

comparison_restaurants = pd.DataFrame(rows)
comparison_restaurants = comparison_restaurants.sort_values(
    ["cuisine_type", "city", "review_count", "stars"],
    ascending=[True, True, False, False],
).reset_index(drop=True)

print(f"Comparison restaurants found : {len(comparison_restaurants)}")
print()
print("Counts by cuisine:")
print(comparison_restaurants["cuisine_type"].value_counts().to_string())
print()
print("Counts by city:")
print(comparison_restaurants.groupby(["cuisine_type", "city"]).size().to_string())
print()
print(comparison_restaurants.head(5).to_string())

## 5. Save `comparison_restaurants.parquet` and run gate checks

This cell writes the comparison restaurant slice to disk and checks the obvious failure modes: empty cuisines, missing metros, and suspiciously low row counts.

In [ ]:
assert len(comparison_restaurants) > 0, "No comparison restaurants found."
assert set(comparison_restaurants["cuisine_type"]) == set(CUISINE_MAP), (
    "One or more target cuisines is missing from the comparison restaurant set."
)
assert set(comparison_restaurants["city"]) == set(METROS), (
    "One or more target metros is missing from the comparison restaurant set."
)

COMPARISON_RESTAURANTS_OUT.parent.mkdir(parents=True, exist_ok=True)
comparison_restaurants.to_parquet(COMPARISON_RESTAURANTS_OUT, index=False)

gate_targets = {
    "american_new_restaurants": len(comparison_restaurants[comparison_restaurants["cuisine_type"] == "american_new"]) >= 900,
    "mexican_restaurants":      len(comparison_restaurants[comparison_restaurants["cuisine_type"] == "mexican"]) >= 400,
}

print(f"Saved: {COMPARISON_RESTAURANTS_OUT.relative_to(ROOT)}")
print(f"Rows : {len(comparison_restaurants)}")
print()
print("Phase 1c restaurant gates:")
for label, ok in gate_targets.items():
    print(f"  {label:<28} [{'OK' if ok else 'CHECK'}]")
print()
print("File size MB:", round(COMPARISON_RESTAURANTS_OUT.stat().st_size / 1e6, 3))

## 6. Extract comparison reviews and cap to 200 most recent per restaurant

The Yelp review file is too large to load eagerly, so this cell streams it once, keeps only reviews for the comparison restaurants, sorts them by date within each restaurant, and trims to the most recent `REVIEW_CAP` rows.

In [ ]:
comparison_ids = set(comparison_restaurants["business_id"])
city_lookup    = comparison_restaurants.set_index("business_id")["city"].to_dict()
cuisine_lookup = comparison_restaurants.set_index("business_id")["cuisine_type"].to_dict()

review_rows = []
with open(YELP_REVIEWS, encoding="utf-8") as f:
    for line in tqdm(f, desc="Filtering reviews", unit=" rows"):
        rec = json.loads(line)
        bid = rec.get("business_id")
        if bid not in comparison_ids:
            continue
        review_rows.append({
            "review_id":    rec.get("review_id"),
            "business_id":  bid,
            "user_id":      rec.get("user_id"),
            "stars":        rec.get("stars"),
            "text":         rec.get("text"),
            "date":         rec.get("date"),
            "useful":       rec.get("useful"),
            "funny":        rec.get("funny"),
            "cool":         rec.get("cool"),
            "city":         city_lookup[bid],
            "cuisine_type": cuisine_lookup[bid],
        })

comparison_reviews = pd.DataFrame(review_rows)
comparison_reviews["date"] = pd.to_datetime(comparison_reviews["date"], errors="coerce")
comparison_reviews = comparison_reviews.sort_values(
    ["business_id", "date"], ascending=[True, False]
)
comparison_reviews = (
    comparison_reviews.groupby("business_id", group_keys=False)
    .head(REVIEW_CAP)
    .reset_index(drop=True)
)

print(f"Comparison reviews retained : {len(comparison_reviews)}")
print()
print("Counts by cuisine:")
print(comparison_reviews["cuisine_type"].value_counts().to_string())
print()
print("Per-business review cap check:")
print(comparison_reviews.groupby("business_id").size().describe().to_string())

## 7. Save `comparison_reviews.parquet` and verify the cap

This cell persists the trimmed comparison reviews and checks that no business exceeds the 200-review limit.

In [ ]:
assert len(comparison_reviews) > 0, "No comparison reviews found."
assert comparison_reviews.groupby("business_id").size().max() <= REVIEW_CAP, (
    "Review cap failed: at least one business exceeds REVIEW_CAP."
)

comparison_reviews.to_parquet(COMPARISON_REVIEWS_OUT, index=False)

review_gate_targets = {
    "american_new_reviews": len(comparison_reviews[comparison_reviews["cuisine_type"] == "american_new"]) >= 100_000,
    "mexican_reviews":      len(comparison_reviews[comparison_reviews["cuisine_type"] == "mexican"]) >= 40_000,
}

print(f"Saved: {COMPARISON_REVIEWS_OUT.relative_to(ROOT)}")
print(f"Rows : {len(comparison_reviews)}")
print()
print("Phase 1c review gates:")
for label, ok in review_gate_targets.items():
    print(f"  {label:<28} [{'OK' if ok else 'CHECK'}]")
print()
print("Date span:")
print(f"  {comparison_reviews['date'].min()}  ->  {comparison_reviews['date'].max()}")

## 8. Build the comparison Yelp menu-photo manifest

This cell scans Yelp's `photos.json` and keeps only `label == "menu"` rows for the comparison restaurants. It does not extract files yet; it just builds the manifest we will use in the next step.

In [ ]:
ensure_photo_manifest()

photo_rows = []
with open(PHOTO_JSON, encoding="utf-8") as f:
    for line in tqdm(f, desc="Reading photo manifest", unit=" rows"):
        rec = json.loads(line)
        bid = rec.get("business_id")
        if bid not in comparison_ids:
            continue
        if rec.get("label") != "menu":
            continue
        photo_rows.append({
            "photo_id":      rec.get("photo_id"),
            "business_id":   bid,
            "caption":       rec.get("caption"),
            "label":         rec.get("label"),
            "cuisine_type":  cuisine_lookup[bid],
        })

comparison_photo_manifest = pd.DataFrame(photo_rows)

print(f"Comparison menu-labelled photos : {len(comparison_photo_manifest)}")
print()
if len(comparison_photo_manifest):
    print(comparison_photo_manifest["cuisine_type"].value_counts().to_string())
    print()
    print(comparison_photo_manifest.head(10).to_string(index=False))
else:
    print("No comparison Yelp menu photos found. The extraction cell will still run safely.")

## 9. Extract the comparison Yelp menu photos

Menu-labelled Yelp photos are copied into `data/raw/menu_images_comparison/` with one directory per cuisine and business. The cell only extracts files that are not already present, so re-runs are safe.

In [ ]:
COMPARISON_PHOTO_ROOT.mkdir(parents=True, exist_ok=True)

expected = {}
for row in comparison_photo_manifest.to_dict("records"):
    tar_path = f"photos/{row['photo_id']}.jpg"
    dest_dir = COMPARISON_PHOTO_ROOT / row["cuisine_type"] / row["business_id"]
    expected[tar_path] = dest_dir / f"{row['photo_id']}.jpg"

already_present = {str(p.relative_to(COMPARISON_PHOTO_ROOT)) for p in COMPARISON_PHOTO_ROOT.rglob("*.jpg")}
remaining = {
    tar_path: dest for tar_path, dest in expected.items()
    if str(dest.relative_to(COMPARISON_PHOTO_ROOT)) not in already_present
}

print(f"Already present : {len(expected) - len(remaining)}")
print(f"To extract      : {len(remaining)}")

if remaining:
    with tarfile.open(PHOTO_TAR, "r:gz") as tf:
        members = {m.name: m for m in tf.getmembers()}
        for tar_path, dest in tqdm(remaining.items(), desc="Extracting comparison photos"):
            member = members.get(tar_path)
            if member is None:
                continue
            dest.parent.mkdir(parents=True, exist_ok=True)
            data = tf.extractfile(member).read()
            dest.write_bytes(data)

final_photos = sorted(COMPARISON_PHOTO_ROOT.rglob("*.jpg"))
print(f"Final extracted photos : {len(final_photos)}")
if final_photos:
    by_cuisine = defaultdict(int)
    for p in final_photos:
        cuisine = p.relative_to(COMPARISON_PHOTO_ROOT).parts[0]
        by_cuisine[cuisine] += 1
    print("Counts by cuisine:")
    for cuisine, n in sorted(by_cuisine.items()):
        print(f"  {cuisine:<13} {n}")

## 10. Stage the Roboflow OCR_Menu corpus

This project only needs a verified local copy of the Roboflow dataset. To keep the notebook practical rather than brittle, this cell supports two simple paths:

1. `data/raw/roboflow/ocr_menu/` already exists
2. `data/raw/roboflow/ocr_menu.zip` exists and should be extracted

If neither is present, the cell stops with a short instruction rather than trying to guess a Roboflow API configuration.

In [ ]:
ROBOFLOW_ROOT.parent.mkdir(parents=True, exist_ok=True)

if ROBOFLOW_ROOT.exists():
    print(f"Roboflow root already present: {ROBOFLOW_ROOT}")
elif ROBOFLOW_ZIP.exists():
    print(f"Extracting {ROBOFLOW_ZIP.name} ...")
    with zipfile.ZipFile(ROBOFLOW_ZIP) as zf:
        zf.extractall(ROBOFLOW_ROOT.parent)
    if not ROBOFLOW_ROOT.exists():
        raise FileNotFoundError(
            "Zip extracted, but data/raw/roboflow/ocr_menu/ was not created. "
            "Check the archive layout and move the extracted folder into place."
        )
else:
    raise FileNotFoundError(
        "Roboflow corpus not found. Place the extracted dataset at "
        "data/raw/roboflow/ocr_menu/ or add ocr_menu.zip to data/raw/roboflow/."
    )

roboflow_images = collect_image_paths(ROBOFLOW_ROOT)
assert len(roboflow_images) > 0, "Roboflow root exists but no image files were found."

print(f"Roboflow images found : {len(roboflow_images)}")
for p in roboflow_images[:10]:
    print(f"  {p.relative_to(ROOT)}")

## 11. Phase 1c verification summary

This is the acquisition gate for the combined notebook. It verifies row counts, photo availability, and whether the Roboflow image files are readable by PIL.

In [ ]:
roboflow_meta, roboflow_unreadable = scan_image_metadata(
    roboflow_images,
    source="roboflow",
    cuisine_type="american_new",
)

mean_roboflow_mpx = (
    np.mean([(row["width"] * row["height"]) / 1e6 for row in roboflow_meta])
    if roboflow_meta else 0.0
)

print("=" * 64)
print("PHASE 1c SUMMARY — Comparison acquisition")
print("=" * 64)
print(f"Comparison restaurants     : {len(comparison_restaurants):>7}")
print(f"Comparison reviews         : {len(comparison_reviews):>7}")
print(f"Comparison menu photos     : {len(list(COMPARISON_PHOTO_ROOT.rglob('*.jpg'))):>7}")
print(f"Roboflow images            : {len(roboflow_meta):>7}")
print(f"Roboflow unreadable        : {len(roboflow_unreadable):>7}")
print(f"Roboflow mean resolution   : {mean_roboflow_mpx:.3f} Mpx")
print()
print("Gate checks:")
checks = {
    "american_new restaurants >= 900": len(comparison_restaurants[comparison_restaurants["cuisine_type"] == "american_new"]) >= 900,
    "mexican restaurants >= 400":      len(comparison_restaurants[comparison_restaurants["cuisine_type"] == "mexican"]) >= 400,
    "american_new reviews >= 100000":  len(comparison_reviews[comparison_reviews["cuisine_type"] == "american_new"]) >= 100_000,
    "mexican reviews >= 40000":        len(comparison_reviews[comparison_reviews["cuisine_type"] == "mexican"]) >= 40_000,
    "Roboflow readable by PIL":        len(roboflow_unreadable) == 0,
    "Roboflow mean >= 0.3 Mpx":        mean_roboflow_mpx >= 0.3,
}
for label, ok in checks.items():
    print(f"  {label:<34} [{'OK' if ok else 'CHECK'}]")

if roboflow_unreadable:
    print("\nUnreadable Roboflow files (first 5):")
    for path, err in roboflow_unreadable[:5]:
        print(f"  {path.relative_to(ROOT)} :: {err}")

## 12. LanceDB rebuild setup

The next cells rebuild the `restaurants` and `reviews` tables with a unified `cuisine_type` field. This is the start of the old NB01d workflow.

In [ ]:
assert MODEL_PATH.exists(), f"Embedding model not found: {MODEL_PATH}"

DB_PATH.mkdir(parents=True, exist_ok=True)
db = lancedb.connect(str(DB_PATH))

print(f"DB path        : {DB_PATH}")
print(f"Existing tables: {db.table_names() if db.table_names() else '(none)'}")
print(f"Embedding model: {MODEL_PATH.name}")

## 13. Load the embedding model and probe the vector size

Only one model is loaded in this notebook rebuild phase. This cell should use roughly the same ~700 MB VRAM observed in Phase 0.

In [ ]:
print("Loading embedding model...")
embedder = Llama(
    model_path=str(MODEL_PATH),
    embedding=True,
    n_ctx=N_CTX,
    n_gpu_layers=N_GPU_LAYERS,
    verbose=False,
)

probe = embedder.embed("menuforge probe")
EMBED_DIM = len(probe)
print(f"Embedding dimension : {EMBED_DIM}")
print(f"Vector norm         : {np.linalg.norm(probe):.4f}")

## 14. Rebuild the unified `restaurants` table

This step merges the Italian baseline with the two comparison cuisines, embeds the rows, and recreates the LanceDB table with FTS indexes on the main retrieval fields.

In [ ]:
italian_restaurants = pd.read_parquet(ITALIAN_RESTAURANTS).copy()
italian_restaurants["cuisine_type"] = "italian"

comparison_restaurants = pd.read_parquet(COMPARISON_RESTAURANTS_OUT).copy()

restaurant_columns = sorted(
    set(italian_restaurants.columns).union(comparison_restaurants.columns)
)
italian_restaurants = italian_restaurants.reindex(columns=restaurant_columns)
comparison_restaurants = comparison_restaurants.reindex(columns=restaurant_columns)

restaurants_all = pd.concat(
    [italian_restaurants, comparison_restaurants],
    ignore_index=True,
)
restaurants_all["embed_text"] = (
    restaurants_all["name"].fillna("") + " " +
    restaurants_all["categories"].fillna("") + " " +
    restaurants_all["city"].fillna("") + " " +
    restaurants_all["cuisine_type"].fillna("")
).str.strip()

print(f"Unified restaurants rows : {len(restaurants_all)}")
print(restaurants_all["cuisine_type"].value_counts().to_string())

In [ ]:
restaurant_vectors = [
    embedder.embed(text)
    for text in tqdm(restaurants_all["embed_text"].tolist(), desc="Embedding restaurants")
]
restaurants_all["vector"] = restaurant_vectors
restaurants_all = restaurants_all.drop(columns=["embed_text"])

TABLE = "restaurants"
if TABLE in db.table_names():
    db.drop_table(TABLE)
    print(f"Dropped existing '{TABLE}' table.")

tbl_restaurants = db.create_table(TABLE, data=restaurants_all)
for field in ["name", "categories", "address", "city", "cuisine_type"]:
    tbl_restaurants.create_fts_index(field, replace=True)

print(f"Table '{TABLE}' rebuilt")
print(f"  Rows    : {tbl_restaurants.count_rows()}")
print(f"  Columns : {tbl_restaurants.schema.names}")

## 15. Rebuild the unified `reviews` table

This is the longest step in the notebook. The first cell prepares the combined review frame and recreates the empty table schema; the second cell embeds and appends review batches.

In [ ]:
italian_reviews = pd.read_parquet(ITALIAN_REVIEWS).copy()
italian_reviews["cuisine_type"] = "italian"

comparison_reviews = pd.read_parquet(COMPARISON_REVIEWS_OUT).copy()

review_columns = sorted(set(italian_reviews.columns).union(comparison_reviews.columns))
italian_reviews = italian_reviews.reindex(columns=review_columns)
comparison_reviews = comparison_reviews.reindex(columns=review_columns)

reviews_all = pd.concat([italian_reviews, comparison_reviews], ignore_index=True)
reviews_all["date"] = pd.to_datetime(reviews_all["date"], errors="coerce")
reviews_all = reviews_all.sort_values(["business_id", "date"], ascending=[True, False]).reset_index(drop=True)

print(f"Unified reviews rows : {len(reviews_all)}")
print(reviews_all["cuisine_type"].value_counts().to_string())

REVIEW_SCHEMA = pa.schema([
    pa.field("review_id",    pa.string()),
    pa.field("business_id",  pa.string()),
    pa.field("user_id",      pa.string()),
    pa.field("stars",        pa.float64()),
    pa.field("text",         pa.string()),
    pa.field("date",         pa.string()),
    pa.field("useful",       pa.int64()),
    pa.field("funny",        pa.int64()),
    pa.field("cool",         pa.int64()),
    pa.field("city",         pa.string()),
    pa.field("cuisine_type", pa.string()),
    pa.field("vector",       pa.list_(pa.float32())),
])

TABLE = "reviews"
if TABLE in db.table_names():
    db.drop_table(TABLE)
    print(f"Dropped existing '{TABLE}' table.")

tbl_reviews = db.create_table(TABLE, schema=REVIEW_SCHEMA)
for field in ["text", "city", "cuisine_type"]:
    tbl_reviews.create_fts_index(field, replace=True)

print(f"Table '{TABLE}' created (empty)")
print(f"  Schema : {tbl_reviews.schema.names}")

In [ ]:
review_records = reviews_all.copy()
review_records["date"] = review_records["date"].dt.strftime("%Y-%m-%d")

for start in tqdm(range(0, len(review_records), REVIEW_BATCH_SIZE), desc="Embedding reviews", unit="batch"):
    chunk = review_records.iloc[start:start + REVIEW_BATCH_SIZE].copy()
    chunk["vector"] = [
        embedder.embed(text)
        for text in chunk["text"].fillna("").tolist()
    ]
    tbl_reviews.add(chunk)

total_rows = tbl_reviews.count_rows()
print(f"Done. Table rows : {total_rows}")
assert total_rows == len(review_records), (
    f"Review row mismatch: expected {len(review_records)}, got {total_rows}"
)

## 16. Create the `american_menus` metadata table

This table stores metadata only, not embeddings. It combines Roboflow menu images with American New Yelp menu photos so Phase 3 can query a single American corpus.

In [ ]:
american_yelp_rows = []
american_yelp_unreadable = []

for biz_dir in sorted((COMPARISON_PHOTO_ROOT / "american_new").glob("*")):
    if not biz_dir.is_dir():
        continue
    rows, unreadable = scan_image_metadata(
        sorted(biz_dir.glob("*.jpg")),
        source="yelp",
        cuisine_type="american_new",
        business_id=biz_dir.name,
    )
    american_yelp_rows.extend(rows)
    american_yelp_unreadable.extend(unreadable)

roboflow_rows, roboflow_bad = scan_image_metadata(
    roboflow_images,
    source="roboflow",
    cuisine_type="american_new",
)

american_menu_meta = pd.DataFrame(american_yelp_rows + roboflow_rows)

TABLE = "american_menus"
if TABLE in db.table_names():
    db.drop_table(TABLE)
    print(f"Dropped existing '{TABLE}' table.")

tbl_american = db.create_table(TABLE, data=american_menu_meta)
for field in ["source", "cuisine_type", "business_id", "filename"]:
    if field in american_menu_meta.columns:
        tbl_american.create_fts_index(field, replace=True)

print(f"Table '{TABLE}' created")
print(f"  Rows         : {tbl_american.count_rows()}")
print(f"  Yelp rows    : {len(american_yelp_rows)}")
print(f"  Roboflow rows: {len(roboflow_rows)}")
print(f"  Unreadable   : {len(american_yelp_unreadable) + len(roboflow_bad)}")

## 17. Confirm unchanged tables from NB01b

`nypl_menus` and `menu_items` should already exist from NB01b. This cell does not modify them; it simply confirms their status so the rebuild is transparent.

In [ ]:
for name in ["nypl_menus", "menu_items"]:
    if name not in db.table_names():
        print(f"{name:<12} MISSING — run NB01b if this table is required.")
        continue
    table = db.open_table(name)
    print(f"{name:<12} rows={table.count_rows():>7}")

## 18. Unified rebuild gate checks

This is the final verification pass for the combined notebook: row counts, cuisine coverage, `american_menus` presence, and a few retrieval sanity checks.

In [ ]:
tables = {name: db.open_table(name) for name in db.table_names()}

restaurants_check = tables["restaurants"].to_pandas()
reviews_check     = tables["reviews"].to_pandas()

print("=" * 66)
print("PHASE 1d SUMMARY — Unified LanceDB rebuild")
print("=" * 66)
print("Restaurants by cuisine:")
print(restaurants_check["cuisine_type"].value_counts().to_string())
print()
print("Reviews by cuisine:")
print(reviews_check["cuisine_type"].value_counts().to_string())
print()
print(f"american_menus rows : {tables['american_menus'].count_rows() if 'american_menus' in tables else 0}")

print("\nGate checks:")
gate_checks = {
    "italian restaurants >= 717":      (restaurants_check["cuisine_type"] == "italian").sum() >= 717,
    "american_new restaurants >= 900": (restaurants_check["cuisine_type"] == "american_new").sum() >= 900,
    "mexican restaurants >= 400":      (restaurants_check["cuisine_type"] == "mexican").sum() >= 400,
    "italian reviews >= 105774":       (reviews_check["cuisine_type"] == "italian").sum() >= 105_774,
    "american_new reviews >= 100000":  (reviews_check["cuisine_type"] == "american_new").sum() >= 100_000,
    "mexican reviews >= 40000":        (reviews_check["cuisine_type"] == "mexican").sum() >= 40_000,
    "american_menus table exists":     "american_menus" in tables,
}
for label, ok in gate_checks.items():
    print(f"  {label:<32} [{'OK' if ok else 'CHECK'}]")

print("\nRetrieval sanity checks:")
test_queries = [
    ("pasta", "italian"),
    ("burger", "american_new"),
    ("taco", "mexican"),
]
for query, cuisine in test_queries:
    try:
        res = (
            tables["restaurants"]
            .search(query)
            .where(f"cuisine_type = '{cuisine}'", prefilter=True)
            .limit(3)
            .to_pandas()
        )
        ok = len(res) > 0
        print(f"  {query:<10} in {cuisine:<13} [{'OK' if ok else 'CHECK'}]  rows={len(res)}")
        if ok:
            print(res[["name", "city", "cuisine_type"]].to_string(index=False))
    except Exception as e:
        print(f"  {query:<10} in {cuisine:<13} [CHECK]  {e}")

## 19. Close-out summary

This final card records what the notebook produced and points directly to the next EDA notebooks that now become possible (`NB02e`, `NB02f`, `NB02g`).

In [ ]:
print("=" * 68)
print("NB01cd COMPLETE — Three-cuisine acquisition + unified rebuild")
print("=" * 68)
print("\nFiles written:")
print(f"  {COMPARISON_RESTAURANTS_OUT.relative_to(ROOT)}")
print(f"  {COMPARISON_REVIEWS_OUT.relative_to(ROOT)}")
print(f"  {COMPARISON_PHOTO_ROOT.relative_to(ROOT)}")
print(f"  {DB_PATH.relative_to(ROOT)}")

print("\nWhat is now ready:")
print("  - NB02e retrieval comparison on the unified restaurants table")
print("  - NB02f three-way cuisine comparison EDA")
print("  - NB02g American menu-image inventory")

print("\nNotes:")
print("  - The review embedding step is the slowest part of this notebook.")
print("  - Roboflow is staged locally rather than downloaded via a brittle API guess.")
print("  - nypl_menus and menu_items remain under NB01b ownership unless you choose otherwise.")